## Driver Standings View

Ranks all drivers by season based on total points and wins.

---

#### Sources

| Table | Provides |
| --- | --- |
| `formula1.gold.facts_session_results` | Points, wins, podiums |
| `formula1.gold.dim_drivers` | Driver name, nationality |

---

#### Logic

1. **Join** fact table with driver dimension on `driver_id`
2. **Aggregate** per driver per season
3. **Rank** using `RANK()` by points then wins (descending)

---

#### Output - `formula1.gold.v_driver_standings`

| Column | Description |
| --- | --- |
| `season` | Championship year |
| `driver_id` | Driver identifier |
| `driver_name` | Full name |
| `nationality` | Driver nationality |
| `race_starts` | Sessions entered |
| `total_points` | Points scored |
| `number_of_wins` | P1 finishes |
| `number_of_podiums` | Top-3 finishes |
| `standing` | Championship position |

In [0]:
Create or REPLACE view formula1.gold.v_driver_standings
AS
With driver_session_summary
as
(select r.season,
       d.driver_id,
       d.driver_name,
       d.nationality,
       count(*) AS race_starts,
       sum(r.points) AS total_points,
       count_if(r.is_win) AS number_of_wins,
       count_if(r.is_podium) AS number_of_podiums
from formula1.gold.facts_session_results r
join formula1.gold.dim_drivers d
on r.driver_id = d.driver_id
GROUP BY r.season,
       d.driver_id,
       d.driver_name,
       d.nationality)
select season,
       driver_id,
       driver_name,
       nationality,
       race_starts,
       total_points,
       number_of_wins,
       number_of_podiums,
       RANK() OVER(PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing
from driver_session_summary

#### Verify - 2025 Season Standings
Query the view filtered to the current season to confirm standings are calculated correctly.

In [0]:
select * from formula1.gold.v_driver_standings where season = 2025;